In [1]:
import os
from pprint import pprint
from pathlib import Path
from copy import deepcopy

import numpy as np
from scipy.constants import pi
import matplotlib.pyplot as plt
from matplotlib import colors as mcolors
from matplotlib import rcParams
rcParams["font.size"] = 18

from quvac.postprocess import integrate_spherical
from quvac.plotting import plot_mollweide
from quvac.utils import write_yaml, read_yaml

SCRIPT_PATH = '../src/quvac/simulation.py'
PARALLEL_PATH = '../src/quvac/simulation_parallel.py'
GRIDSCAN_PATH = '../src/quvac/cluster/gridscan.py'
OPTIMIZATION_PATH = '../src/quvac/cluster/optimization.py'

path = "../data/tutorials/tutorial_4"
Path(path).mkdir(parents=True, exist_ok=True)

# choose 'local' or 'slurm' here
CLUSTER_TYPE = 'local'
SLURM_PARTITION = 'cpu'

## Light-by-light scenario

In what follows, we would demonstrate the usage of parallel simulations, gridscan and optimization. All of these examples would be based on two counter-propagating gaussian setup, parameters of which we define here.

In [2]:
lam = 800e-9
w0 = 2 * lam
tau = 25e-15

# all parameters in SI units
field_1_params = {
    "field_type": "paraxial_gaussian_maxwell",
    "focus_x": [0.,0.,0.],
    "focus_t": 0.,
    "theta": 0,
    "phi": 0,
    "beta": 0,
    "phase0": 0,
    "lam": lam,
    "w0": w0,
    "tau": tau,
    "W": 25,
    "order": 0,
}

# add counter-propagating field
# (it has the same parameters apart from propagation direction and polarization)
field_2_params = deepcopy(field_1_params)
field_2_params["theta"] = 180
field_2_params["beta"] = 45

# combine
fields_params = {
    "field_1": field_1_params,
    "field_2": field_2_params,
}

# grid parameters
grid_params = {
    'mode': 'dynamic',
    'collision_geometry': 'z',
    'transverse_factor': 20,
    'longitudinal_factor': 8,
    'time_factor': 4,
    'spatial_resolution': 1,
    'time_resolution': 2,
}

ini_data = {
    'mode': 'simulation_postprocess',
    'fields': fields_params,
    'grid': grid_params,
    'integrator': {
        'type': 'vacuum_emission',
    },
}

## Parallel simulations

quvac can run a simulation in parallel by splitting the discretized time integral between different jobs and then collecting results together. A separate script and command is responsible for such computation. To control the parameters of parallelization (number of jobs, parameters of each job), a separate section in `ini.yml` is required.

Usually such parallelization makes sense only on cluster enviroments.

We would calculate the same counter-propagating setup.

The parallel calculation is launched on the slurm cluster (this is controlled by `cluster`: 'local' or 'slurm'). Provide necessary sbatch parameters (they are similar to stardard sbatch scripts) which could be looked up as possible keyword arguments from [submitit](https://github.com/facebookincubator/submitit).

In [3]:
# cluster parameters <---- NEW
cluster_params = {
    'number_of_time_intervals': 4,
    'cluster_type': CLUSTER_TYPE,
    'sbatch_params': {
        'slurm_partition': SLURM_PARTITION,
        'cpus_per_task': 2,
        'memory': '1GB',
        'walltime': "1:00:00",
    },
}

ini_data_local = deepcopy(ini_data)
ini_data_local['cluster_params'] = cluster_params

folder_seq = "sequential"
folder_par = "parallel"    

In [4]:
def launch_simulation(ini_data_local, folder, number_of_time_intervals):
    ini_data_local['cluster_params']['number_of_time_intervals'] = number_of_time_intervals
    
    path_local = os.path.join(path, folder)
    Path(path_local).mkdir(parents=True, exist_ok=True)
    ini_file = os.path.join(path_local, 'ini.yml')
    write_yaml(ini_file, ini_data_local)

    status = os.system(f"{PARALLEL_PATH} --input {ini_file}")

If the cost of field initialization is neglidgeable compared to the vacuum emission amplitude calculation then the speedup is typically around number of jobs. One should take into account that disk memory usage also grows with the number of jobs.

In [5]:
launch_simulation(ini_data_local, folder_seq, number_of_time_intervals=1)


Timings:
Run jobs:                    1 min 36.09 s
Postprocess:                        0.04 s
----------------------------------------------------
Total:                       1 min 36.13 s

Memory (max usage):
Running jobs:                    117.21 MB
Total:                           132.98 MB

Simulation finished!


In [6]:
launch_simulation(ini_data_local, folder_par, number_of_time_intervals=4)


Timings:
Run jobs:                    4 min 35.83 s
Postprocess:                        0.03 s
----------------------------------------------------
Total:                       4 min 35.86 s

Memory (max usage):
Running jobs:                    117.20 MB
Total:                           133.38 MB

Simulation finished!


## Gridscan

For the gridscan script we need to specify the parameters for which the grid would be created (as well as grid properties).
Then the script creates grids for requested parameters, writes `ini.yml` file for each simulation and does the job submission with `submitit`. 

Here we consider the same two gaussian counter-propagating setup and make a gridscan over polarization orientation of one of the beams ($\beta$).

In [13]:
def launch_gridscan(ini_data_local, folder):
    path_local = os.path.join(path, folder)
    Path(path_local).mkdir(parents=True, exist_ok=True)
    ini_file = os.path.join(path_local, 'ini.yml')
    write_yaml(ini_file, ini_data_local)

    status = os.system(f"{GRIDSCAN_PATH} --input {ini_file}")

In [20]:
# gridscan parameters <---- NEW
variables_data = {
    'create_grids': True,
    'fields': {
        'field_2': {
            'beta': [0, 90, 5]
        }
    },
    'cluster_params': {
        'cluster_type': CLUSTER_TYPE,
        'max_parallel_jobs': 2,
        'sbatch_params': {
            'slurm_partition': SLURM_PARTITION,
            'cpus_per_task': 1,
            'memory': '1GB',
            'walltime': '1:00:00',   
        },
    },
}

ini_data_local = deepcopy(ini_data)
ini_data_local["variables"] = variables_data

folder = "gridscan"

In [ ]:
launch_gridscan(ini_data_local, folder)

### Plot results

In [18]:
beta_start, beta_end, n_pts = variables_data["fields"]["field_2"]["beta"]
betas = np.linspace(beta_start, beta_end, n_pts, dtype=int)
N_totals = []

for beta in betas:
    path_local = os.path.join(path, folder, f"#field_2:beta_{beta}")
    data = np.load(os.path.join(path_local, "spectra_total.npz"))
    N_totals.append(data["N_total"])

FileNotFoundError: [Errno 2] No such file or directory: '../data/tutorials/tutorial_4/gridscan/#field_2:beta_0/spectra_total.npz'

In [ ]:
plt.figure()
plt.plot(betas, N_totals, ".--", ms=15)
plt.grid()
plt.xticks(betas)
plt.xlabel("$\\beta_2$")
plt.ylabel("$N_{total}$")
plt.show()

Traceback (most recent call last):
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/home/maximus/micromamba/envs/quvac-new/lib/python3.12/site-packages/submitit/local/_local.py", line 16, in <module>
    controller.run()
  File "/home/maximus/micromamba/envs/quvac-new/lib/python3.12/site-packages/submitit/local/local.py", line 375, in run
    exit_codes = self.wait()
                 ^^^^^^^^^^^
  File "/home/maximus/micromamba/envs/quvac-new/lib/python3.12/site-packages/submitit/local/local.py", line 366, in wait
    time.sleep(1.0 / freq)
KeyboardInterrupt
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "<frozen runpy>", line 88, in _run_code
  File "/home/maximus/micromamba/envs/quvac-new/lib/pyt

The first beam's polarization is aligned along $x$ ($\beta_1 = 0^{\circ}$), $\beta_2$ is the polarization of the second beam. We know that the maximal total signal is achieved for the polarization difference $\Delta \beta = 90^{\circ}$ which agrees with the plot above.

## Optimization

Optimization script requires its own set of parameters defining both optimization procedure parameters and sbatch specifications for each job (similar to gridscan). The Bayesian optimization is performed with `Ax` library and job submission is taken care by `submitit`.

Here we would optimize the polarization alignment of the 2nd beam ($\beta_2$) to maximize the total signal.

In [ ]:
def launch_optimization(ini_data_local, folder):
    path_local = os.path.join(path, folder)
    Path(path_local).mkdir(parents=True, exist_ok=True)
    ini_file = os.path.join(path_local, 'ini.yml')
    write_yaml(ini_file, ini_data_local)

    status = os.system(f"{OPTIMIZATION_PATH} --input {ini_file}")

In [ ]:
# optimization parameters <---- NEW
optimization_data = {
    'name': 'beta_2',
    'parameters': {
        'field_2': {"beta": [0,90],}
    },
    'cluster_params': {
        'cluster_type': CLUSTER_TYPE,
        'max_parallel_jobs': 3,
        'sbatch_params': {
            "slurm_partition": SLURM_PARTITION,
            "cpus_per_task": 2,
            "memory": "2GB",
            "walltime": "1:00:00",   
        },
    },
    'n_trials': 7,
    'objective': 'N_total',
    'generation_strategy_type': 'quality',
}

ini_data_local = deepcopy(ini_data)
ini_data_local["optimization"] = optimization_data
folder = "optimization"

In [ ]:
launch_optimization(ini_data_local, folder)

### Plot results

In [ ]:
from ax.api.client import Client
from quvac.cluster.optimization import gather_trials_data

client_json = os.path.join(path, folder, 'experiment.json')
ax_client = Client.load_from_json_file(client_json)
trials_params = gather_trials_data(ax_client, metric_names=['N_total'])

print(trials_params)

In [ ]:
def get_data(trials, key):
    vals = [val[key] for val in trials_params.values()]
    return np.array(vals)

betas = get_data(trials_params, "field_2:beta")
N_totals = get_data(trials_params, "N_total")

idx_best = np.argmax(N_totals)
beta_best = betas[idx_best]
N_total_best = N_totals[idx_best]

In [ ]:
plt.figure()
plt.scatter(betas, N_totals, color='green', edgecolor='black', marker='s',
            zorder=2, label='trials')
plt.plot(beta_best, N_total_best, "*", color='red', ms=12, label='best')
plt.grid()
plt.xticks([10*i for i in range(10)])
plt.xlabel("$\\beta_2$")
plt.ylabel("$N_{total}$")
plt.legend()
plt.show()

Traceback (most recent call last):
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/home/maximus/micromamba/envs/quvac-new/lib/python3.12/site-packages/submitit/local/_local.py", line 16, in <module>
Traceback (most recent call last):
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
Traceback (most recent call last):
  File "/home/maximus/micromamba/envs/quvac-new/lib/python3.12/site-packages/submitit/local/_local.py", line 16, in <module>
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/home/maximus/micromamba/envs/quvac-new/lib/python3.12/site-packages/submitit/local/_local.py", line 16, in <module>
    controller.run()
  File "/home/maximus/micromamba/envs/quvac-new/lib/python3.12/site-packages/submitit/local/local.py", line 375, in run
    controller.run()
    controller.run()
  File "/home/maximus/

As we see, the optimization found the same optimum.

In [ ]:
from collections.abc import Iterable
from ax.adapter.registry import Generators
from ax.core.observation import ObservationFeatures
import itertools

class SurrogateModel:
    def __init__(self, ax_client, metric="N_total"):
        self.experiment_data = ax_client._experiment.fetch_data()
        self.model = Generators.BOTORCH_MODULAR(
            experiment=ax_client._experiment,
            data=self.experiment_data,
        )
        self.metric = metric

    def _update_input_parameters(model_input, fixed_params):
        if fixed_parameters:
            for point in model_input:
                point.update(fixed_parameters)
        model_input = [ObservationFeatures(parameters=pt) for pt in model_input]
        return model_input
    
    def _make_model_prediction(self, model_input, fixed_params):
        model_input = self._update_input_parameters(model_input, fixed_params)
        mean, covariance = self.model.predict(model_input)
        mean, covariance = np.array(mean[self.metric]), np.array(covariance[self.metric][self.metric])
        return mean, covariance

    def predict_at_point(param_name, param_value, fixed_params=None):
        model_input = [{param_name: param_value}]
        mean, covariance = self._make_model_prediction(model_input, fixed_params)
        return mean, covariance

    def predict_1d(param_name, param_values, fixed_params=None):
        assert isinstance(param_values, Iterable), "Method `predict_1d` works only with sequences"
        model_input = [{param_name: value} for value in param_values]
        mean, covariance = self._make_model_prediction(model_input, fixed_params)
        return mean, covariance

    def predict_2d(param_names, param_values, fixed_params=None):
        assert len(param_names) == 2, "Only two variable parameters are accepted"
        assert len(param_values) == 2, "Only two variable parameter grids are accepted"
        param_name_1, param_name_2 = param_names
        n1, n2 = len(param_values[0]), len(param_values[1])
        model_input = [
            {param_name_1: value1, param_name_2: value2}
            for value1,value2 in itertools.product(*param_values)
        ]
        mean, covariance = self._make_model_prediction(model_input, fixed_params)
        mean, covariance = mean.reshape((n1,n2)), covariance.reshape((n1,n2))
        return mean, covariance

In [ ]:
model = SurrogateModel(ax_client)

betas_to_predict = np.linspace(0, 90, 37)
param_name = "field_2:beta"

mean, cov = model.predict_1d(param_name, betas_to_predict)

In [ ]:
plt.figure()
plt.plot(betas_to_predict, mean, "-", color="tab:green", ms=12)
plt.fill_between(betas_to_predict, mean-cov, mean+cov, color="tab:green", alpha=0.5)
plt.scatter(betas, N_totals, color='green', edgecolor='black', marker='s',
            zorder=2, label='trials')
plt.grid()
plt.xticks([10*i for i in range(10)])
plt.xlabel("$\\beta_2$")
plt.ylabel("$N_{total}$")
plt.show()